In [0]:
-- Create the Gold schema if it does not exist
CREATE SCHEMA IF NOT EXISTS weather_openmeteo.gold;

use weather_openmeteo.gold;

-- Create the KPIs target table with the specific data types
CREATE TABLE IF NOT EXISTS weather_openmeteo.gold.weather_daily_kpis (
    date TIMESTAMP,
    latitude DOUBLE,
    longitude DOUBLE,
    city STRING,
    temperature_2m_max DOUBLE,
    temperature_2m_min DOUBLE,
    daylight_duration DOUBLE,
    wind_speed_10m_max DOUBLE,
    avg_temperature_7d DOUBLE,
    alerts_temp BOOLEAN,
    alerts_wind BOOLEAN
)
USING DELTA;

-- Calculate KPIs from the Silver table using Window functions
WITH gold_kpis AS (
    SELECT 
        date,
        latitude,
        longitude,
        city,
        temperature_2m_max,
        temperature_2m_min,
        daylight_duration,
        wind_speed_10m_max,
        
        -- Calculates the average temperature for the current day and the next 6 days (7 days total)
        AVG(temperature_2m_max) OVER (
            PARTITION BY city, latitude, longitude
            ORDER BY CAST(date AS DATE)
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS avg_temperature_7d,
        
        /*
        Weather conditions are generally considered in the "danger zone" when temperatures exceed a Heat Index of 103ºF or fall below a Wind Chill of -20ºF.
        */
        -- Returns true if the temperature exceeds 103°F or is below -20°F; otherwise, returns false.
        CASE 
            WHEN temperature_2m_max <-20.0000 or temperature_2m_max > 90.0000 THEN true 
            ELSE false 
        END AS alerts_temp,
        
        /*
        The National Weather Service issues advisories and warnings based on wind thresholds: 
        Wind Advisory (26-39 mph sustained / 35-57 mph gusts): Use extreme caution.
        High Wind Warning (40+ mph sustained / 58+ mph gusts): Avoid driving altogether.
        */
        -- Returns true if the wind speed exceeds 40.0000, otherwise false
        CASE 
            WHEN wind_speed_10m_max > 40.0000 THEN true 
            ELSE false 
        END AS alerts_wind
    FROM weather_openmeteo.silver.weather_daily_clean
)

-- Execute the Merge operation into the Gold table
MERGE INTO weather_openmeteo.gold.weather_daily_kpis AS target
USING gold_kpis AS source
ON target.date = source.date 
   AND target.latitude = source.latitude 
   AND target.longitude = source.longitude

-- When the record already exists, update the KPIs and metrics
WHEN MATCHED THEN
  UPDATE SET
    target.city = source.city,
    target.temperature_2m_max = source.temperature_2m_max,
    target.temperature_2m_min = source.temperature_2m_min,
    target.daylight_duration = source.daylight_duration,
    target.wind_speed_10m_max = source.wind_speed_10m_max,
    target.avg_temperature_7d = CAST(source.avg_temperature_7d AS DOUBLE),
    target.alerts_temp = source.alerts_temp,
    target.alerts_wind = source.alerts_wind

-- When the record is new, insert it into the Gold table
WHEN NOT MATCHED THEN
  INSERT (date, latitude, longitude, city, temperature_2m_max, temperature_2m_min, daylight_duration, wind_speed_10m_max, avg_temperature_7d, alerts_temp, alerts_wind)
  VALUES (source.date, source.latitude, source.longitude, source.city, source.temperature_2m_max, source.temperature_2m_min, source.daylight_duration, source.wind_speed_10m_max, CAST(source.avg_temperature_7d AS DOUBLE), source.alerts_temp, source.alerts_wind);

 -- Display the Gold table
SELECT * FROM weather_openmeteo.gold.weather_daily_kpis

